In [4]:
%pip install pandas numpy scikit-learn matplotlib seaborn imbalanced-learn xgboost lightgbm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    precision_score, recall_score, accuracy_score,
    roc_auc_score, roc_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print('Libraries loaded successfully.')


[notice] A new release of pip is available: 23.3.1 -> 26.0.1
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Libraries loaded successfully.


In [5]:
df = pd.read_csv('../data/collisions_clean.csv')
df['CRASH DATE'] = pd.to_datetime(df['CRASH DATE'], errors='coerce')
df['hour'] = pd.to_numeric(df['hour'], errors='coerce')

# Target
df['target_severity'] = df['severity'].apply(lambda x: 1 if x in ['Injury', 'Fatal'] else 0)

# Time interval
def time_interval(hour):
    if pd.isna(hour): return 'Unknown'
    hour = int(hour)
    if 6 <= hour <= 9: return 'Morning Rush'
    elif 10 <= hour <= 15: return 'Midday'
    elif 16 <= hour <= 19: return 'Evening Rush'
    elif 20 <= hour <= 23: return 'Night'
    else: return 'Late Night'

df['time_interval'] = df['hour'].apply(time_interval)

# Fill NaN
df['BOROUGH'] = df['BOROUGH'].fillna('Unknown')
df['vehicle_type_clean'] = df['vehicle_type_clean'].fillna('Unknown')
df['factor_category'] = df['factor_category'].fillna('Unknown')
df['season'] = df['season'].fillna('Unknown')

# Encode
cat_cols = ['BOROUGH', 'vehicle_type_clean', 'factor_category', 'season', 'time_interval']
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Train/test split
temporal_cutoff = pd.to_datetime('2024-01-01')
train_df = df_encoded[df_encoded['CRASH DATE'] < temporal_cutoff].copy()
test_df = df_encoded[df_encoded['CRASH DATE'] >= temporal_cutoff].copy()

# Feature columns
exclude_cols = [
    'CRASH DATE', 'severity', 'target_severity',
    'total_casualties', 'has_injury', 'has_fatality',
    'has_pedestrian_casualty', 'has_cyclist_casualty',
    'NUMBER OF PERSONS INJURED', 'NUMBER OF PERSONS KILLED',
    'NUMBER OF PEDESTRIANS INJURED', 'NUMBER OF PEDESTRIANS KILLED',
    'NUMBER OF CYCLIST INJURED', 'NUMBER OF CYCLIST KILLED',
    'NUMBER OF MOTORIST INJURED', 'NUMBER OF MOTORIST KILLED',
    'LATITUDE', 'LONGITUDE', 'primary_factor',
]

feature_cols = [c for c in df_encoded.columns
                if c not in exclude_cols
                and df_encoded[c].dtype in ['int64', 'float64', 'uint8', 'bool']]

X_train = train_df[feature_cols].fillna(0)
X_test = test_df[feature_cols].fillna(0)
y_train = train_df['target_severity']
y_test = test_df['target_severity']

print(f'Train: {X_train.shape[0]:,} records, {X_train.shape[1]} features')
print(f'Test:  {X_test.shape[0]:,} records, {X_test.shape[1]} features')
print(f'Train target rate: {y_train.mean()*100:.1f}%')
print(f'Test target rate:  {y_test.mean()*100:.1f}%')

Train: 192,037 records, 47 features
Test:  182,988 records, 47 features
Train target rate: 40.2%
Test target rate:  43.8%


In [6]:
class_counts = y_train.value_counts()
print("Class distribution:", class_counts.to_dict())

Class distribution: {0: 114849, 1: 77188}


In [7]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# ----- Use original data (NO SMOTE) -----
X_tr_sm, y_tr_sm = X_train, y_train

# ----- Compute class imbalance weight -----
neg = len(y_tr_sm) - y_tr_sm.sum()
pos = y_tr_sm.sum()

scale_pos_weight = neg / pos

print(f"scale_pos_weight: {scale_pos_weight:.4f}")

# ----- Train XGBoost -----
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    tree_method='hist',        # faster
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight
)

xgb.fit(X_tr_sm, y_tr_sm)

# ----- Predictions -----
xgb_preds = xgb.predict(X_test)
xgb_probs = xgb.predict_proba(X_test)[:, 1]

# ----- Evaluation -----
xgb_results = {
    "Accuracy": accuracy_score(y_test, xgb_preds),
    "Precision": precision_score(y_test, xgb_preds),
    "Recall": recall_score(y_test, xgb_preds),
    "F1": f1_score(y_test, xgb_preds),
    "ROC_AUC": roc_auc_score(y_test, xgb_probs)
}

print("\nXGBoost Results (Weighted, No SMOTE):")
for k, v in xgb_results.items():
    print(f"{k}: {v:.4f}")

scale_pos_weight: 1.4879

XGBoost Results (Weighted, No SMOTE):
Accuracy: 0.6446
Precision: 0.5914
Recall: 0.6087
F1: 0.5999
ROC_AUC: 0.7020


In [8]:
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Compute class imbalance weight
neg = len(y_train) - y_train.sum()
pos = y_train.sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight: {scale_pos_weight:.4f}")

# Train LightGBM
lgb_clf = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    scale_pos_weight=scale_pos_weight
)

lgb_clf.fit(X_train, y_train)

# Predictions
lgb_preds = lgb_clf.predict(X_test)
lgb_probs = lgb_clf.predict_proba(X_test)[:, 1]

# Evaluation
lgb_results = {
    "Accuracy": accuracy_score(y_test, lgb_preds),
    "Precision": precision_score(y_test, lgb_preds),
    "Recall": recall_score(y_test, lgb_preds),
    "F1": f1_score(y_test, lgb_preds),
    "ROC_AUC": roc_auc_score(y_test, lgb_probs)
}

print("\nLightGBM Results (Weighted, No SMOTE):")
for k, v in lgb_results.items():
    print(f"{k}: {v:.4f}")

scale_pos_weight: 1.4879
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 77188, number of negative: 114849
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002964 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 339
[LightGBM] [Info] Number of data points in the train set: 192037, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.401943 -> initscore=-0.397374
[LightGBM] [Info] Start training from score -0.397374

LightGBM Results (Weighted, No SMOTE):
Accuracy: 0.6431
Precision: 0.5904
Recall: 0.6029
F1: 0.5965
ROC_AUC: 0.7017


In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.pipeline import make_pipeline

# ----- Create a pipeline with scaling + Logistic Regression -----
lr_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        random_state=42,
        max_iter=1000,       # increase if it doesn't converge
        class_weight='balanced'  # handles class imbalance
    )
)

# ----- Fit the model -----
lr_pipeline.fit(X_train, y_train)

# ----- Predictions -----
lr_preds = lr_pipeline.predict(X_test)
lr_probs = lr_pipeline.predict_proba(X_test)[:, 1]

# ----- Evaluation -----
lr_results = {
    "Accuracy": accuracy_score(y_test, lr_preds),
    "Precision": precision_score(y_test, lr_preds),
    "Recall": recall_score(y_test, lr_preds),
    "F1": f1_score(y_test, lr_preds),
    "ROC_AUC": roc_auc_score(y_test, lr_probs)
}

print("\nLogistic Regression Results (Scaled Features):")
for k, v in lr_results.items():
    print(f"{k}: {v:.4f}")


Logistic Regression Results (Scaled Features):
Accuracy: 0.6035
Precision: 0.5345
Recall: 0.7297
F1: 0.6170
ROC_AUC: 0.6860


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd

# 1️⃣ Train a base Random Forest
rf_base = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_base.fit(X_train, y_train)

# 2️⃣ Get feature importances
feature_cols = X_train.columns  # make sure feature_cols is defined
rf_importance = pd.Series(rf_base.feature_importances_, index=feature_cols).sort_values(ascending=False)

print('TOP 20 FEATURES (Random Forest importance):')
print(rf_importance.head(20))

# 3️⃣ Select features above threshold
importance_threshold = 0.005
important_features = rf_importance[rf_importance > importance_threshold].index.tolist()
dropped_features = rf_importance[rf_importance <= importance_threshold].index.tolist()

print(f'Features above threshold ({importance_threshold}): {len(important_features)}')
print(f'Features dropped: {len(dropped_features)}')
print(f'Dropped: {dropped_features}')

# 4️⃣ Keep only selected features
X_train_selected = X_train[important_features]
X_test_selected = X_test[important_features]

# 5️⃣ Retrain Random Forest on selected features
rf_selected = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_selected.fit(X_train_selected, y_train)

# 6️⃣ Predictions and evaluation
y_pred_rf_sel = rf_selected.predict(X_test_selected)
y_prob_rf_sel = rf_selected.predict_proba(X_test_selected)[:, 1]

print(f'\nRandom Forest with feature selection:')
print(f'  Features: {X_train.shape[1]} -> {len(important_features)}')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred_rf_sel):.4f}')
print(f'  Precision: {precision_score(y_test, y_pred_rf_sel):.4f}')
print(f'  Recall:    {recall_score(y_test, y_pred_rf_sel):.4f}')
print(f'  F1:        {f1_score(y_test, y_pred_rf_sel):.4f}')
print(f'  ROC AUC:   {roc_auc_score(y_test, y_prob_rf_sel):.4f}')

TOP 20 FEATURES (Random Forest importance):
ZIP CODE                                     0.174705
hour                                         0.157688
day_of_week                                  0.131558
month                                        0.115084
num_vehicles                                 0.059085
year                                         0.029517
factor_category_Failure to Yield             0.023218
vehicle_type_clean_SUV/Station Wagon         0.016333
is_weekend                                   0.015844
vehicle_type_clean_Sedan                     0.014889
season_Summer                                0.014426
vehicle_type_clean_E-Bike/E-Scooter          0.014051
time_interval_Midday                         0.012923
season_Winter                                0.012865
season_Spring                                0.012072
BOROUGH_BROOKLYN                             0.011437
factor_category_Distracted Driving           0.010752
factor_category_Improper Lane Use/Pass

In [11]:
import pandas as pd

# Combine all results
results_df = pd.DataFrame({
    "XGBoost": xgb_results,
    "LightGBM": lgb_results,
    "Logistic Regression": lr_results,
    "Random Forest (FS)": {
        "Accuracy": accuracy_score(y_test, y_pred_rf_sel),
        "Precision": precision_score(y_test, y_pred_rf_sel),
        "Recall": recall_score(y_test, y_pred_rf_sel),
        "F1": f1_score(y_test, y_pred_rf_sel),
        "ROC_AUC": roc_auc_score(y_test, y_prob_rf_sel)
    }
})

# Transpose for better readability
results_df = results_df.T
print(results_df)

                     Accuracy  Precision    Recall        F1   ROC_AUC
XGBoost              0.644649   0.591354  0.608713  0.599908  0.701960
LightGBM             0.643113   0.590369  0.602857  0.596548  0.701658
Logistic Regression  0.603510   0.534452  0.729669  0.616987  0.685971
Random Forest (FS)   0.617838   0.584580  0.438236  0.500938  0.638740


In [13]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
import pandas as pd
import numpy as np

# ----- Datasets -----
X_lr = X_train.copy()            # scaled features for Logistic Regression
X_rf = X_train_selected.copy()   # feature-selected dataset for Random Forest
X_full = X_train.copy()          # full features for XGBoost and LightGBM
y_cv = y_train.copy()

# ----- Compute class imbalance weight for tree models -----
neg = len(y_cv) - y_cv.sum()
pos = y_cv.sum()
scale_pos_weight = neg / pos

# ----- Models -----
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            random_state=42,
            max_iter=1000,
            class_weight='balanced',  # handle imbalance
            n_jobs=-1
        )
    ),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        tree_method='hist',
        random_state=42,
        eval_metric='logloss',
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1
    ),
    "LightGBM": lgb.LGBMClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1
    )
}

# ----- Stratified CV -----
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# ----- Metrics -----
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

# ----- Run CV -----
cv_results = {}
for name, model in models.items():
    # assign proper dataset for each model
    if name == "Logistic Regression":
        X_use = X_lr
    elif name == "Random Forest":
        X_use = X_rf
    else:
        X_use = X_full

    scores = cross_validate(model, X_use, y_cv, cv=cv, scoring=scoring, n_jobs=-1, return_train_score=False)
    cv_results[name] = {metric: (np.mean(scores[f'test_{metric}']), np.std(scores[f'test_{metric}'])) for metric in scoring}

# ----- Train models on full data to get feature importance -----
top_features = {}
for name, model in models.items():
    if name == "Logistic Regression":
        model.fit(X_lr, y_cv)
        # coefficients after StandardScaler
        coefs = model.named_steps['logisticregression'].coef_[0]
        feat_importance = pd.Series(np.abs(coefs), index=X_lr.columns)
    elif name == "Random Forest":
        model.fit(X_rf, y_cv)
        feat_importance = pd.Series(model.feature_importances_, index=X_rf.columns)
    elif name == "XGBoost":
        model.fit(X_full, y_cv)
        feat_importance = pd.Series(model.feature_importances_, index=X_full.columns)
    else:  # LightGBM
        model.fit(X_full, y_cv)
        feat_importance = pd.Series(model.feature_importances_, index=X_full.columns)
    
    # get top 3 features
    top_features[name] = feat_importance.sort_values(ascending=False).head(3).index.tolist()

# ----- Convert results to DataFrame -----
rows = []
for model_name, scores in cv_results.items():
    row = {"Model": model_name, "Top Features": ", ".join(top_features[model_name])}
    for metric, (mean, std) in scores.items():
        row[metric.capitalize()] = f"{mean:.4f} ± {std:.4f}"
    rows.append(row)

cv_df = pd.DataFrame(rows)
cv_df

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 77188, number of negative: 114849
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003178 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 339
[LightGBM] [Info] Number of data points in the train set: 192037, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.401943 -> initscore=-0.397374
[LightGBM] [Info] Start training from score -0.397374


,Model,Top Features,Accuracy,Precision,Recall,F1,Roc_auc
0,Logistic Regression,"vehicle_type_clean_Sedan, vehicle_type_clean_S...",0.6526 ± 0.0014,0.5716 ± 0.0017,0.5421 ± 0.0025,0.5565 ± 0.0021,0.6952 ± 0.0019
1,Random Forest,"ZIP CODE, hour, day_of_week",0.6305 ± 0.0007,0.5525 ± 0.0011,0.4247 ± 0.0020,0.4802 ± 0.0014,0.6423 ± 0.0016
2,XGBoost,"vehicle_type_clean_E-Bike/E-Scooter, factor_ca...",0.6553 ± 0.0025,0.5708 ± 0.0032,0.5742 ± 0.0020,0.5725 ± 0.0026,0.7043 ± 0.0022
3,LightGBM,"num_vehicles, ZIP CODE, hour",0.6553 ± 0.0022,0.5705 ± 0.0027,0.5759 ± 0.0032,0.5732 ± 0.0030,0.7045 ± 0.0021


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 51458, number of negative: 76566
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002218 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 332
[LightGBM] [Info] Number of data points in the train set: 128024, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.401940 -> initscore=-0.397387
[LightGBM] [Info] Start training from score -0.397387
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 51459, number of negative: 76566
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001990 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_c

In [35]:
import numpy as np
import pandas as pd

# Already trained models
# lr_pipeline -> Logistic Regression pipeline
# lgb_clf -> LightGBM classifier

# Use the same features as for training
X_use = X_train.copy()

# Get predictions from both models
lr_probs = lr_pipeline.predict_proba(X_use)[:,1]
lgb_probs = lgb_clf.predict_proba(X_use)[:,1]

# --- Define a rule to choose which model to trust ---
# Simple heuristic: if LR predicts high confidence (prob near 0 or 1), trust LR
# otherwise trust LightGBM (nonlinear interactions)
confidence_threshold = 0.6

hybrid_preds = []
for lr_p, lgb_p in zip(lr_probs, lgb_probs):
    if lr_p > confidence_threshold:
        hybrid_preds.append(1)
    elif lr_p < (1-confidence_threshold):
        hybrid_preds.append(0)
    else:
        # LR unsure, use LightGBM
        hybrid_preds.append(int(lgb_p > 0.5))

hybrid_preds = np.array(hybrid_preds)

# --- Evaluate ---
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Use probabilities for ROC AUC
hybrid_probs = np.array([
    lr_p if lr_p > confidence_threshold or lr_p < (1-confidence_threshold) else lgb_p
    for lr_p, lgb_p in zip(lr_probs, lgb_probs)
])

hybrid_results = {
    "Accuracy": accuracy_score(y_train, hybrid_preds),
    "Precision": precision_score(y_train, hybrid_preds),
    "Recall": recall_score(y_train, hybrid_preds),
    "F1": f1_score(y_train, hybrid_preds),
    "ROC_AUC": roc_auc_score(y_train, hybrid_probs)  # <-- use probs here
}

print("Hybrid LR + LightGBM Results:")
for k,v in hybrid_results.items():
    print(f"{k}: {v:.4f}")

Hybrid LR + LightGBM Results:
Accuracy: 0.6643
Precision: 0.5812
Recall: 0.5894
F1: 0.5853
ROC_AUC: 0.7108


In [37]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Already trained models
# lr_pipeline -> Logistic Regression pipeline
# lgb_clf -> LightGBM classifier

# Parameters
confidence_threshold = 0.6
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# To store metrics for each fold
metrics = {"Accuracy": [], "Precision": [], "Recall": [], "F1": [], "ROC_AUC": []}

X_use = X_train.copy()
y_use = y_train.copy()

for train_idx, val_idx in skf.split(X_use, y_use):
    X_tr, X_val = X_use.iloc[train_idx], X_use.iloc[val_idx]
    y_tr, y_val = y_use.iloc[train_idx], y_use.iloc[val_idx]
    
    # Get predictions from already trained models
    lr_probs = lr_pipeline.predict_proba(X_val)[:,1]
    lgb_probs = lgb_clf.predict_proba(X_val)[:,1]
    
    # Hybrid predictions
    hybrid_preds = []
    hybrid_probs = []
    for lr_p, lgb_p in zip(lr_probs, lgb_probs):
        if lr_p > confidence_threshold:
            hybrid_preds.append(1)
            hybrid_probs.append(lr_p)
        elif lr_p < (1 - confidence_threshold):
            hybrid_preds.append(0)
            hybrid_probs.append(lr_p)
        else:
            hybrid_preds.append(int(lgb_p > 0.5))
            hybrid_probs.append(lgb_p)
    
    hybrid_preds = np.array(hybrid_preds)
    hybrid_probs = np.array(hybrid_probs)
    
    # Compute metrics
    metrics["Accuracy"].append(accuracy_score(y_val, hybrid_preds))
    metrics["Precision"].append(precision_score(y_val, hybrid_preds))
    metrics["Recall"].append(recall_score(y_val, hybrid_preds))
    metrics["F1"].append(f1_score(y_val, hybrid_preds))
    metrics["ROC_AUC"].append(roc_auc_score(y_val, hybrid_probs))

# Print mean results across folds
print("Hybrid LR + LightGBM CV Results (5-fold):")
for k, v in metrics.items():
    print(f"{k}: {np.mean(v):.4f} ± {np.std(v):.4f}")

Hybrid LR + LightGBM CV Results (5-fold):
Accuracy: 0.6643 ± 0.0015
Precision: 0.5812 ± 0.0022
Recall: 0.5894 ± 0.0024
F1: 0.5853 ± 0.0016
ROC_AUC: 0.7108 ± 0.0013
